# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset title: {metadata.name}")
print(f"Description: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

We enumerate the record sets, fields in each, and their columns with their unique `@id` values.

In [ ]:
# List all record sets' @id and show field details
print("Available record sets and their fields:")
record_set_ids = []
for rs in metadata.record_sets:
    print(f"  RecordSet @id: {rs.id}  Name: {rs.name}")
    record_set_ids.append(rs.id)
    for field in rs.fields:
        print(f"    Field @id: {field.id}, Name: {field.name}, Data type: {field.data_type}")
        # Try to display columns for the field if present
        if hasattr(field, 'columns') and field.columns:
            for col in field.columns:
                print(f"      Column @id: {col.id}, Name: {col.name}")

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. Use the record set and field `@id`s found above.

In [ ]:
# Extract all records for each record set and load into DataFrames
dataframes = dict()
for record_set_id in record_set_ids:
    print(f"Loading records for RecordSet @id: {record_set_id}")
    rows = list(dataset.records(record_set=record_set_id))
    if not rows:
        print(f"  No records found for {record_set_id}.")
        continue
    df = pd.DataFrame(rows)
    dataframes[record_set_id] = df
    print(f"  Columns for {record_set_id}: {df.columns.tolist()}")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps like filtering, normalizing, and grouping in a chosen record set (pick a numeric field, e.g., GAD-7 or PHQ-9 score).

In [ ]:
# Example: work with the main survey record set -- find the GAD-7 score field
# Please adjust these variables based on section 2 if your field/recordset IDs differ

# Select the main survey record set (by inspection; often the first/only main set)
if not record_set_ids:
    raise ValueError("No record sets found in metadata.")
record_set_id = record_set_ids[0]  # Change index if needed for your main data
df = dataframes[record_set_id]

# Identify a numeric field used in EDA (assume it's called 'gad7_score' or similar after code cell 2 output, adjust if needed)
numeric_field = None
for col in df.columns:
    if (
        col.lower().startswith('gad7') 
        or 'gad' in col.lower() and 'score' in col.lower()
        or col.lower().startswith('phq9')
    ):
        numeric_field = col
        break
# If not found, fall back to a field by manual inspection
if numeric_field is None:
    print("Did not auto-detect GAD-7/PHQ-9 score column; using the first numeric-looking field.")
    for col in df.select_dtypes(include='number').columns:
        numeric_field = col
        break

if numeric_field is None:
    raise ValueError("No numeric field found for EDA.")

print(f"Using numeric field: {numeric_field}")
threshold = df[numeric_field].mean()  # example threshold; can adjust

# Filtering
filtered_df = df[df[numeric_field] > threshold].copy()
print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
print(filtered_df.head())

# Normalizing
filtered_df[f"{numeric_field}_normalized"] = (
    filtered_df[numeric_field] - filtered_df[numeric_field].mean()
) / (filtered_df[numeric_field].std() if filtered_df[numeric_field].std() > 0 else 1)
print(f"Normalized {numeric_field} for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Grouping: try a likely group field, e.g., 'gender' or 'village'. Adjust if necessary.
group_field = None
for col in df.columns:
    if col.lower() in ['gender', 'sex', 'village', 'location']:
        group_field = col
        break
# If not found, use the first categorical-looking field
if group_field is None:
    for col in df.columns:
        if df[col].dtype == object and df[col].nunique() < 20:
            group_field = col
            break

if group_field:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
    print(f"Grouped data by {group_field} (mean {numeric_field}):")
    print(grouped_df)

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the chosen numeric field
plt.figure(figsize=(8, 4))
sns.histplot(df[numeric_field].dropna(), bins=20, kde=True)
plt.title(f"Distribution of {numeric_field}")
plt.xlabel(numeric_field)
plt.ylabel('Count')
plt.show()

# If group_field is present, boxenplot by group
if group_field:
    plt.figure(figsize=(8, 5))
    sns.boxenplot(x=group_field, y=numeric_field, data=df)
    plt.title(f"{numeric_field} by {group_field}")
    plt.xticks(rotation=30)
    plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The FAIR² Kilifi Mental Health Survey dataset was loaded successfully via the mlcroissant library.
- Key record sets, fields, and data types were identified programmatically via their Croissant `@id`s.
- Basic statistics and distributions for important mental health measures (e.g., GAD-7, PHQ-9) were visualized.
- Further analysis can be performed by referencing variables and record sets using their canonical Croissant `@id` identifiers for reproducibility and FAIR compliance.